In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, lower, round, current_date, lit
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = SparkSession.builder.appName("Day3").getOrCreate()

data = [
    ("Ravi",   "Engineering", "Pune",      "55000", "M", "28", "2021-03-15"),
    ("Priya",  "HR",          "Mumbai",    "42000", "F", "32", "2019-07-22"),
    ("Arjun",  "Engineering", "Delhi",     "72000", "M", "26", "2022-01-10"),
    ("Sneha",  "Finance",     "Pune",      "61000", "F", "30", "2020-11-05"),
    ("Rohit",  "Engineering", "Mumbai",    "80000", "M", "35", "2018-06-30"),
    ("Meera",  "HR",          "Bangalore", "39000", "F", "27", "2023-02-18"),
    ("Karan",  "Finance",     "Delhi",     "55000", "M", "29", "2021-09-12"),
    ("Divya",  "Engineering", "Pune",      "91000", "F", "33", "2017-04-25"),
]

cols = ["name", "dept", "city", "salary", "gender", "age", "joining_date"]

# Notice: salary and age are STRING here on purpose — for casting practice
df = spark.createDataFrame(data, cols)
df.printSchema()   # notice salary = string, age = string
df.show()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- joining_date: string (nullable = true)

+-----+-----------+---------+------+------+---+------------+
| name|       dept|     city|salary|gender|age|joining_date|
+-----+-----------+---------+------+------+---+------------+
| Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
|Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
|Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
|Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
|Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
|Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|
|Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
|Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
+-----+-----------+---------+------+------+---+------------+

In [3]:
df.select(col("joining_date").alias ("JoinDate")).show()

+----------+
|  JoinDate|
+----------+
|2021-03-15|
|2019-07-22|
|2022-01-10|
|2020-11-05|
|2018-06-30|
|2023-02-18|
|2021-09-12|
|2017-04-25|
+----------+



In [4]:
from pyspark.sql.functions import col

df.select(col("joining_date").alias("JoinDate")).show()

+----------+
|  JoinDate|
+----------+
|2021-03-15|
|2019-07-22|
|2022-01-10|
|2020-11-05|
|2018-06-30|
|2023-02-18|
|2021-09-12|
|2017-04-25|
+----------+



In [5]:
df

DataFrame[name: string, dept: string, city: string, salary: string, gender: string, age: string, joining_date: string]

In [6]:
df.show()

+-----+-----------+---------+------+------+---+------------+
| name|       dept|     city|salary|gender|age|joining_date|
+-----+-----------+---------+------+------+---+------------+
| Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
|Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
|Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
|Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
|Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
|Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|
|Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
|Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
+-----+-----------+---------+------+------+---+------------+



In [7]:
df.printschema()    # spelling mistake

AttributeError: 'DataFrame' object has no attribute 'printschema'

In [8]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- joining_date: string (nullable = true)



In [9]:
# 1a. Add a brand new column — annual salary
df = df.withColumn("annual_salary", col("salary") * 12)
# ❌ This will FAIL — why? Because salary is StringType!
# You'll fix this in Task 2 (casting)


In [10]:
df.select("name", "salary", "annual_salary").groupBy("annual_salary").show(3)   ## spark did implicit casting

AttributeError: 'GroupedData' object has no attribute 'show'

In [11]:
df.groupBy("annual_salary").count().show()

+-------------+-----+
|annual_salary|count|
+-------------+-----+
|     660000.0|    2|
|     504000.0|    1|
|     864000.0|    1|
|     732000.0|    1|
|     960000.0|    1|
|     468000.0|    1|
|    1092000.0|    1|
+-------------+-----+



In [12]:
df.groupBy().agg(avg("salary")).show()

NameError: name 'avg' is not defined

In [13]:
from pyspark.sql.functions import avg

df.groupBy().agg(avg("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    61875.0|
+-----------+



In [14]:
from pyspark.sql.functions import sum

df.groupBy().sum("annual_salary").show()

+------------------+
|sum(annual_salary)|
+------------------+
|         5940000.0|
+------------------+



The real issue

groupBy() does not return a DataFrame that you can display directly.

It returns a GroupedData object, which must be followed by an aggregation like:

count()
sum()
avg()
max()
min()

In [15]:
df.show(3)

+-----+-----------+------+------+------+---+------------+-------------+
| name|       dept|  city|salary|gender|age|joining_date|annual_salary|
+-----+-----------+------+------+------+---+------------+-------------+
| Ravi|Engineering|  Pune| 55000|     M| 28|  2021-03-15|     660000.0|
|Priya|         HR|Mumbai| 42000|     F| 32|  2019-07-22|     504000.0|
|Arjun|Engineering| Delhi| 72000|     M| 26|  2022-01-10|     864000.0|
+-----+-----------+------+------+------+---+------------+-------------+
only showing top 3 rows



In [16]:
# 1b. Add a constant column
df.withColumn("Country").show()

TypeError: DataFrame.withColumn() missing 1 required positional argument: 'col'

Why?

The syntax of withColumn() is:

withColumn(column_name, column_expression)
It always requires two arguments:

Name of the new (or existing) column
The value or expression for that column

In [17]:
from pyspark.sql.functions import lit

df = df.withColumn("Country", lit("India")).show()

+-----+-----------+---------+------+------+---+------------+-------------+-------+
| name|       dept|     city|salary|gender|age|joining_date|annual_salary|Country|
+-----+-----------+---------+------+------+---+------------+-------------+-------+
| Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|     660000.0|  India|
|Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|     504000.0|  India|
|Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|     864000.0|  India|
|Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|     732000.0|  India|
|Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|     960000.0|  India|
|Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|     468000.0|  India|
|Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|     660000.0|  India|
|Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|    1092000.0|  India|
+-----+-----------+---------+------+------+---+------------+-------------+-------+



In [18]:
df = df.withColumn("NewSalary", col("salary") + 5000)

AttributeError: 'NoneType' object has no attribute 'withColumn'

In [19]:
print(df)

None


show() displays the DataFrame but returns None.

In [21]:
df.printSchema()

AttributeError: 'NoneType' object has no attribute 'printSchema'

In [23]:
df.show(3)  #After running dataframe cell once again

+-----+-----------+------+------+------+---+------------+
| name|       dept|  city|salary|gender|age|joining_date|
+-----+-----------+------+------+------+---+------------+
| Ravi|Engineering|  Pune| 55000|     M| 28|  2021-03-15|
|Priya|         HR|Mumbai| 42000|     F| 32|  2019-07-22|
|Arjun|Engineering| Delhi| 72000|     M| 26|  2022-01-10|
+-----+-----------+------+------+------+---+------------+
only showing top 3 rows



In [24]:
df = df.withColumn("Country", lit("India"))

In [25]:
df.show(3)

+-----+-----------+------+------+------+---+------------+-------+
| name|       dept|  city|salary|gender|age|joining_date|Country|
+-----+-----------+------+------+------+---+------------+-------+
| Ravi|Engineering|  Pune| 55000|     M| 28|  2021-03-15|  India|
|Priya|         HR|Mumbai| 42000|     F| 32|  2019-07-22|  India|
|Arjun|Engineering| Delhi| 72000|     M| 26|  2022-01-10|  India|
+-----+-----------+------+------+------+---+------------+-------+
only showing top 3 rows



In [26]:
df = df.withColumn("New_salary", col("salary") + 45000)

In [27]:
df.show(3)

+-----+-----------+------+------+------+---+------------+-------+----------+
| name|       dept|  city|salary|gender|age|joining_date|Country|New_salary|
+-----+-----------+------+------+------+---+------------+-------+----------+
| Ravi|Engineering|  Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|
|Priya|         HR|Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|
|Arjun|Engineering| Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|
+-----+-----------+------+------+------+---+------------+-------+----------+
only showing top 3 rows



In [28]:
# 1c. Modify existing column — uppercase name


df = df.withColumn("name", upper(col("name")))
df.show(3)

+-----+-----------+------+------+------+---+------------+-------+----------+
| name|       dept|  city|salary|gender|age|joining_date|Country|New_salary|
+-----+-----------+------+------+------+---+------------+-------+----------+
| RAVI|Engineering|  Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|
|PRIYA|         HR|Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|
|ARJUN|Engineering| Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|
+-----+-----------+------+------+------+---+------------+-------+----------+
only showing top 3 rows



In [29]:
# 1d. Add column based on condition
from pyspark.sql.functions import when

df= df.withColumn("salary_Grade", 
    when(col("salary") > 70000 , "High")
    .when(col("salary") > 50000, "Medium")
    .otherwise("Low")
)
df.show()

# Note: comparing strings here — fix after casting

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+
| name|       dept|     city|salary|gender|age|joining_date|Country|New_salary|salary_Grade|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|
|ROHIT|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|  India|  125000.0|        High|
|MEERA|         HR|Bangalore| 39000|     F| 27|  2023-02-18|  India|   84000.0|         Low|
|KARAN|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|  India|  100000.0|      Medium|
|DIVYA|Engineering|     Pune| 91000|     F| 33|  2017-04-25|  India|  

In [30]:
df = df.withColumn("salary_grade",
    when(col("salary") > "70000", "High")
    .when(col("salary") > "50000", "Medium")
    .otherwise("Low")
)
df.show()

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+
| name|       dept|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|
|ROHIT|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|  India|  125000.0|        High|
|MEERA|         HR|Bangalore| 39000|     F| 27|  2023-02-18|  India|   84000.0|         Low|
|KARAN|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|  India|  100000.0|      Medium|
|DIVYA|Engineering|     Pune| 91000|     F| 33|  2017-04-25|  India|  

In [31]:
## 2a. Cast salary and age from String to correct types
df = df.withColumn("salary", col("salary").cast(IntegerType()))
df = df.withColumn("age",    col("age").cast(IntegerType()))

In [32]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- joining_date: string (nullable = true)
 |-- Country: string (nullable = false)
 |-- New_salary: double (nullable = true)
 |-- salary_grade: string (nullable = false)



Best Practice

Always cast the column first:

df = df.withColumn(
    "salary",
    col("salary").cast("double")
)

In [33]:
df = df.withColumn("salary", col("salary").cast("double"))

In [34]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- joining_date: string (nullable = true)
 |-- Country: string (nullable = false)
 |-- New_salary: double (nullable = true)
 |-- salary_grade: string (nullable = false)



In [35]:
df.show(3)


+-----+-----------+------+-------+------+---+------------+-------+----------+------------+
| name|       dept|  city| salary|gender|age|joining_date|Country|New_salary|salary_grade|
+-----+-----------+------+-------+------+---+------------+-------+----------+------------+
| RAVI|Engineering|  Pune|55000.0|     M| 28|  2021-03-15|  India|  100000.0|      Medium|
|PRIYA|         HR|Mumbai|42000.0|     F| 32|  2019-07-22|  India|   87000.0|         Low|
|ARJUN|Engineering| Delhi|72000.0|     M| 26|  2022-01-10|  India|  117000.0|        High|
+-----+-----------+------+-------+------+---+------------+-------+----------+------------+
only showing top 3 rows



In [36]:
df = df.withColumn("salary", col("salary").cast(IntegerType()))
df = df.withColumn("age",    col("age").cast(IntegerType()))

In [37]:
df

DataFrame[name: string, dept: string, city: string, salary: int, gender: string, age: int, joining_date: string, Country: string, New_salary: double, salary_grade: string]

In [38]:
# 2b. Cast to Double
df = df.withColumn("salary_double", col("salary").cast(DoubleType()))

In [39]:
# 2c. Cast using string shorthand (same result)
df = df.withColumn("salary", col("salary").cast("int"))
df = df.withColumn("age",    col("age").cast("integer"))

In [40]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- joining_date: string (nullable = true)
 |-- Country: string (nullable = false)
 |-- New_salary: double (nullable = true)
 |-- salary_grade: string (nullable = false)
 |-- salary_double: double (nullable = true)



In [41]:
# 2d. Now redo annual salary correctly
df = df.withColumn("annual_salary", col("salary") * 12)
df.show()

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+
| name|       dept|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|salary_double|annual_salary|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|      55000.0|       660000|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|      42000.0|       504000|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|      72000.0|       864000|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|      61000.0|       732000|
|ROHIT|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|  India|  125000.0|        High|      80000.0|       960000|
|MEERA|         HR|Bangalore| 39

In [42]:
# 2e. Round to 2 decimal places
df = df.withColumn("salary_in_lakhs",
    round(col("salary") / 100000, 2))
df.show()

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| name|       dept|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|salary_double|annual_salary|salary_in_lakhs|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|      55000.0|       660000|           0.55|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|      42000.0|       504000|           0.42|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|      72000.0|       864000|           0.72|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|      61000.0|       732000|           0.61|
|ROHIT|Engineering|   Mumbai| 80000|     

In [43]:
# TASK 3 — withColumnRenamed + toDF
# 3a. Rename one column
df = df.withColumnRenamed("dept", "department")
df.show()

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| name| department|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|salary_double|annual_salary|salary_in_lakhs|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|      55000.0|       660000|           0.55|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|      42000.0|       504000|           0.42|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|      72000.0|       864000|           0.72|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|      61000.0|       732000|           0.61|
|ROHIT|Engineering|   Mumbai| 80000|     

In [47]:
# 3b. Rename multiple columns using toDF()
# Useful when you want to rename ALL columns at once
new_col_names = ["emp_name", "department", "emp_city",
                 "emp_salary", "emp_gender", "emp_age",
                 "joining_date", "country", "annual_salary",
                 "salary_grade", "salary_double","annual_salary", "salary_in_lakhs"]

df.toDF(*new_col_names).show()
# Use only when column count matches exactly

+--------+-----------+---------+----------+----------+-------+------------+-------+-------------+------------+-------------+-------------+---------------+
|emp_name| department| emp_city|emp_salary|emp_gender|emp_age|joining_date|country|annual_salary|salary_grade|salary_double|annual_salary|salary_in_lakhs|
+--------+-----------+---------+----------+----------+-------+------------+-------+-------------+------------+-------------+-------------+---------------+
|    RAVI|Engineering|     Pune|     55000|         M|     28|  2021-03-15|  India|     100000.0|      Medium|      55000.0|       660000|           0.55|
|   PRIYA|         HR|   Mumbai|     42000|         F|     32|  2019-07-22|  India|      87000.0|         Low|      42000.0|       504000|           0.42|
|   ARJUN|Engineering|    Delhi|     72000|         M|     26|  2022-01-10|  India|     117000.0|        High|      72000.0|       864000|           0.72|
|   SNEHA|    Finance|     Pune|     61000|         F|     30|  2020-1

In [48]:
df.show(truncate=True)

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| name| department|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|salary_double|annual_salary|salary_in_lakhs|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|      55000.0|       660000|           0.55|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|      42000.0|       504000|           0.42|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|      72000.0|       864000|           0.72|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|      61000.0|       732000|           0.61|
|ROHIT|Engineering|   Mumbai| 80000|     

In [49]:
df.show(truncate=15)

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| name| department|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|salary_double|annual_salary|salary_in_lakhs|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+-------------+---------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|      55000.0|       660000|           0.55|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|      42000.0|       504000|           0.42|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|      72000.0|       864000|           0.72|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|      61000.0|       732000|           0.61|
|ROHIT|Engineering|   Mumbai| 80000|     

In [50]:
df.show(vertical=True)

-RECORD 0----------------------
 name            | RAVI        
 department      | Engineering 
 city            | Pune        
 salary          | 55000       
 gender          | M           
 age             | 28          
 joining_date    | 2021-03-15  
 Country         | India       
 New_salary      | 100000.0    
 salary_grade    | Medium      
 salary_double   | 55000.0     
 annual_salary   | 660000      
 salary_in_lakhs | 0.55        
-RECORD 1----------------------
 name            | PRIYA       
 department      | HR          
 city            | Mumbai      
 salary          | 42000       
 gender          | F           
 age             | 32          
 joining_date    | 2019-07-22  
 Country         | India       
 New_salary      | 87000.0     
 salary_grade    | Low         
 salary_double   | 42000.0     
 annual_salary   | 504000      
 salary_in_lakhs | 0.42        
-RECORD 2----------------------
 name            | ARJUN       
 department      | Engineering 
 city   

In [51]:
df.toPandas()

,name,department,city,salary,gender,age,joining_date,Country,New_salary,salary_grade,salary_double,annual_salary,salary_in_lakhs
0,RAVI,Engineering,Pune,55000,M,28,2021-03-15,India,100000.0,Medium,55000.0,660000,0.55
1,PRIYA,HR,Mumbai,42000,F,32,2019-07-22,India,87000.0,Low,42000.0,504000,0.42
2,ARJUN,Engineering,Delhi,72000,M,26,2022-01-10,India,117000.0,High,72000.0,864000,0.72
3,SNEHA,Finance,Pune,61000,F,30,2020-11-05,India,106000.0,Medium,61000.0,732000,0.61
4,ROHIT,Engineering,Mumbai,80000,M,35,2018-06-30,India,125000.0,High,80000.0,960000,0.80
5,MEERA,HR,Bangalore,39000,F,27,2023-02-18,India,84000.0,Low,39000.0,468000,0.39
6,KARAN,Finance,Delhi,55000,M,29,2021-09-12,India,100000.0,Medium,55000.0,660000,0.55
7,DIVYA,Engineering,Pune,91000,F,33,2017-04-25,India,136000.0,High,91000.0,1092000,0.91


In [ ]:
#  toPandas() gives the nicest display for small datasets like your employee table. In real projects with millions of rows, stick to show() 
# because converting large Spark DataFrames to Pandas can consume a lot of memory.

In [52]:
# 3c. Rename using select + alias (cleanest way)
df_renamed = df.select(
    col("name").alias("employee_name"),
    col("department").alias("dept"),
    col("salary").alias("base_salary")
)
df_renamed.show()

+-------------+-----------+-----------+
|employee_name|       dept|base_salary|
+-------------+-----------+-----------+
|         RAVI|Engineering|      55000|
|        PRIYA|         HR|      42000|
|        ARJUN|Engineering|      72000|
|        SNEHA|    Finance|      61000|
|        ROHIT|Engineering|      80000|
|        MEERA|         HR|      39000|
|        KARAN|    Finance|      55000|
|        DIVYA|Engineering|      91000|
+-------------+-----------+-----------+



In [53]:
# ✅ TASK 4 — drop columns
#  Drop one column
df = df.drop("salary_double")
df.show()

+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+---------------+
| name| department|     city|salary|gender|age|joining_date|Country|New_salary|salary_grade|annual_salary|salary_in_lakhs|
+-----+-----------+---------+------+------+---+------------+-------+----------+------------+-------------+---------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  India|  100000.0|      Medium|       660000|           0.55|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|  India|   87000.0|         Low|       504000|           0.42|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  India|  117000.0|        High|       864000|           0.72|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  India|  106000.0|      Medium|       732000|           0.61|
|ROHIT|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|  India|  125000.0|        High|       960000|            0.8|
|MEERA|         

In [54]:
# 4b. Drop multiple columns
df = df.drop("country", "salary_grade")
df.show()

# 4c. Drop using list
cols_to_drop = ["salary_in_lakhs", "annual_salary"]
df = df.drop(*cols_to_drop)
df.show()

+-----+-----------+---------+------+------+---+------------+----------+-------------+---------------+
| name| department|     city|salary|gender|age|joining_date|New_salary|annual_salary|salary_in_lakhs|
+-----+-----------+---------+------+------+---+------------+----------+-------------+---------------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  100000.0|       660000|           0.55|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|       504000|           0.42|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  117000.0|       864000|           0.72|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  106000.0|       732000|           0.61|
|ROHIT|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|  125000.0|       960000|            0.8|
|MEERA|         HR|Bangalore| 39000|     F| 27|  2023-02-18|   84000.0|       468000|           0.39|
|KARAN|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|  100000.0|       6600

#TASK 5 — Date Column Operations

from pyspark.sql.functions import to_date, year, month, datediff, current_date

# 5a. Convert string to date
df = df.withColumn("joining_date", to_date(col("joining_date"), "yyyy-MM-dd"))

# 5b. Extract year and month
df = df.withColumn("join_year",  year(col("joining_date")))
df = df.withColumn("join_month", month(col("joining_date")))

# 5c. Calculate years of experience
df = df.withColumn("experience_days",
    datediff(current_date(), col("joining_date")))

df = df.withColumn("experience_years",
    round(col("experience_days") / 365, 1))

df.select("name", "joining_date", "join_year", "experience_years").show()

In [57]:
from pyspark.sql.functions import to_date, year, month, datediff, current_date

df = df.withColumn("joining_date", to_date(col("joining_date"), "yyyy-MM-dd"))

In [58]:
df

DataFrame[name: string, department: string, city: string, salary: int, gender: string, age: int, joining_date: date, New_salary: double]

In [59]:
df = df.withColumn("join_year", year(col("joining_date")))

In [60]:
df.show(5)

+-----+-----------+------+------+------+---+------------+----------+---------+
| name| department|  city|salary|gender|age|joining_date|New_salary|join_year|
+-----+-----------+------+------+------+---+------------+----------+---------+
| RAVI|Engineering|  Pune| 55000|     M| 28|  2021-03-15|  100000.0|     2021|
|PRIYA|         HR|Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|     2019|
|ARJUN|Engineering| Delhi| 72000|     M| 26|  2022-01-10|  117000.0|     2022|
|SNEHA|    Finance|  Pune| 61000|     F| 30|  2020-11-05|  106000.0|     2020|
|ROHIT|Engineering|Mumbai| 80000|     M| 35|  2018-06-30|  125000.0|     2018|
+-----+-----------+------+------+------+---+------------+----------+---------+
only showing top 5 rows



In [61]:
df = df.withColumn("join_month", month(col("joining_date")))
df.show()

+-----+-----------+---------+------+------+---+------------+----------+---------+----------+
| name| department|     city|salary|gender|age|joining_date|New_salary|join_year|join_month|
+-----+-----------+---------+------+------+---+------------+----------+---------+----------+
| RAVI|Engineering|     Pune| 55000|     M| 28|  2021-03-15|  100000.0|     2021|         3|
|PRIYA|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|     2019|         7|
|ARJUN|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|  117000.0|     2022|         1|
|SNEHA|    Finance|     Pune| 61000|     F| 30|  2020-11-05|  106000.0|     2020|        11|
|ROHIT|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|  125000.0|     2018|         6|
|MEERA|         HR|Bangalore| 39000|     F| 27|  2023-02-18|   84000.0|     2023|         2|
|KARAN|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|  100000.0|     2021|         9|
|DIVYA|Engineering|     Pune| 91000|     F| 33|  2017-04-25|  136000.0

In [62]:
# 5c. Calculate years of experience
df = df.withColumn("Experience_days", datediff(current_date(), col("joining_date")))

In [63]:
df = df.withColumn("experience_years",
    round(col("experience_days") / 365, 1))

df.select("name", "joining_date", "join_year", "experience_years").show()

+-----+------------+---------+----------------+
| name|joining_date|join_year|experience_years|
+-----+------------+---------+----------------+
| RAVI|  2021-03-15|     2021|             5.4|
|PRIYA|  2019-07-22|     2019|             7.0|
|ARJUN|  2022-01-10|     2022|             4.6|
|SNEHA|  2020-11-05|     2020|             5.7|
|ROHIT|  2018-06-30|     2018|             8.1|
|MEERA|  2023-02-18|     2023|             3.5|
|KARAN|  2021-09-12|     2021|             4.9|
|DIVYA|  2017-04-25|     2017|             9.3|
+-----+------------+---------+----------------+



Day 3 Exercises xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
1. Cast age to IntegerType and add column "age_group":
   - "Junior"  if age < 28
   - "Mid"     if age between 28–32
   - "Senior"  if age > 32

2. Add column "salary_category":
   - "Low"    < 45000
   - "Medium" 45000–70000
   - "High"   > 70000

3. Rename "city" to "work_location"

4. Add column "monthly_tax" = 20% of salary (rounded to 2 decimals)

5. Drop the joining_date column after extracting year

6. Add column "name_length" = number of characters in name

In [64]:
df = df.withColumn("age", col("age").cast("int"))

In [65]:
df

DataFrame[name: string, department: string, city: string, salary: int, gender: string, age: int, joining_date: date, New_salary: double, join_year: int, join_month: int, Experience_days: int, experience_years: double]

In [66]:
df = df.withColumn("age_group", 
                   when (col("age") <28, "junior")
                   .when(col("age").between(28,32), "mid")
                   .otherwise("senior"))

In [67]:
df.show(5)

+-----+-----------+------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+
| name| department|  city|salary|gender|age|joining_date|New_salary|join_year|join_month|Experience_days|experience_years|age_group|
+-----+-----------+------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+
| RAVI|Engineering|  Pune| 55000|     M| 28|  2021-03-15|  100000.0|     2021|         3|           1965|             5.4|      mid|
|PRIYA|         HR|Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|     2019|         7|           2567|             7.0|      mid|
|ARJUN|Engineering| Delhi| 72000|     M| 26|  2022-01-10|  117000.0|     2022|         1|           1664|             4.6|   junior|
|SNEHA|    Finance|  Pune| 61000|     F| 30|  2020-11-05|  106000.0|     2020|        11|           2095|             5.7|      mid|
|ROHIT|Engineering|Mumbai| 80000|     M| 35|  2018-06-30|  125000.0| 

In [68]:
df =df.withColumnRenamed("city","work_location")

In [69]:
df.show(5)

+-----+-----------+-------------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+
| name| department|work_location|salary|gender|age|joining_date|New_salary|join_year|join_month|Experience_days|experience_years|age_group|
+-----+-----------+-------------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+
| RAVI|Engineering|         Pune| 55000|     M| 28|  2021-03-15|  100000.0|     2021|         3|           1965|             5.4|      mid|
|PRIYA|         HR|       Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|     2019|         7|           2567|             7.0|      mid|
|ARJUN|Engineering|        Delhi| 72000|     M| 26|  2022-01-10|  117000.0|     2022|         1|           1664|             4.6|   junior|
|SNEHA|    Finance|         Pune| 61000|     F| 30|  2020-11-05|  106000.0|     2020|        11|           2095|             5.7|      mid|
|ROHIT|Engineering| 

In [70]:
df = df.withColumn("monthly_tax", col("salary") * 0.2)

In [71]:
df.show(5)

+-----+-----------+-------------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+-----------+
| name| department|work_location|salary|gender|age|joining_date|New_salary|join_year|join_month|Experience_days|experience_years|age_group|monthly_tax|
+-----+-----------+-------------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+-----------+
| RAVI|Engineering|         Pune| 55000|     M| 28|  2021-03-15|  100000.0|     2021|         3|           1965|             5.4|      mid|    11000.0|
|PRIYA|         HR|       Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|     2019|         7|           2567|             7.0|      mid|     8400.0|
|ARJUN|Engineering|        Delhi| 72000|     M| 26|  2022-01-10|  117000.0|     2022|         1|           1664|             4.6|   junior|    14400.0|
|SNEHA|    Finance|         Pune| 61000|     F| 30|  2020-11-05|  106000.0|     2020|   

In [72]:
df = df.withColumn("monthly_tax", round(col("salary") * 0.20, 2))

In [73]:
df.show(5)

+-----+-----------+-------------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+-----------+
| name| department|work_location|salary|gender|age|joining_date|New_salary|join_year|join_month|Experience_days|experience_years|age_group|monthly_tax|
+-----+-----------+-------------+------+------+---+------------+----------+---------+----------+---------------+----------------+---------+-----------+
| RAVI|Engineering|         Pune| 55000|     M| 28|  2021-03-15|  100000.0|     2021|         3|           1965|             5.4|      mid|    11000.0|
|PRIYA|         HR|       Mumbai| 42000|     F| 32|  2019-07-22|   87000.0|     2019|         7|           2567|             7.0|      mid|     8400.0|
|ARJUN|Engineering|        Delhi| 72000|     M| 26|  2022-01-10|  117000.0|     2022|         1|           1664|             4.6|   junior|    14400.0|
|SNEHA|    Finance|         Pune| 61000|     F| 30|  2020-11-05|  106000.0|     2020|   

In [76]:
df = df.withColumn("join_year", year(col("joining_date"))).drop("joining_date")

In [77]:
df.show(5)

+-----+-----------+-------------+------+------+---+----------+---------+----------+---------------+----------------+---------+-----------+
| name| department|work_location|salary|gender|age|New_salary|join_year|join_month|Experience_days|experience_years|age_group|monthly_tax|
+-----+-----------+-------------+------+------+---+----------+---------+----------+---------------+----------------+---------+-----------+
| RAVI|Engineering|         Pune| 55000|     M| 28|  100000.0|     2021|         3|           1965|             5.4|      mid|    11000.0|
|PRIYA|         HR|       Mumbai| 42000|     F| 32|   87000.0|     2019|         7|           2567|             7.0|      mid|     8400.0|
|ARJUN|Engineering|        Delhi| 72000|     M| 26|  117000.0|     2022|         1|           1664|             4.6|   junior|    14400.0|
|SNEHA|    Finance|         Pune| 61000|     F| 30|  106000.0|     2020|        11|           2095|             5.7|      mid|    12200.0|
|ROHIT|Engineering|       M

In [79]:
df =df.withColumn("name_length", length(col("name")))
df.show(5)

NameError: name 'length' is not defined

In [80]:
from pyspark.sql.functions import length, col
df =df.withColumn("name_length" , length(col("name")))
df.show(5)

+-----+-----------+-------------+------+------+---+----------+---------+----------+---------------+----------------+---------+-----------+-----------+
| name| department|work_location|salary|gender|age|New_salary|join_year|join_month|Experience_days|experience_years|age_group|monthly_tax|name_length|
+-----+-----------+-------------+------+------+---+----------+---------+----------+---------------+----------------+---------+-----------+-----------+
| RAVI|Engineering|         Pune| 55000|     M| 28|  100000.0|     2021|         3|           1965|             5.4|      mid|    11000.0|          4|
|PRIYA|         HR|       Mumbai| 42000|     F| 32|   87000.0|     2019|         7|           2567|             7.0|      mid|     8400.0|          5|
|ARJUN|Engineering|        Delhi| 72000|     M| 26|  117000.0|     2022|         1|           1664|             4.6|   junior|    14400.0|          5|
|SNEHA|    Finance|         Pune| 61000|     F| 30|  106000.0|     2020|        11|           